# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
from openai import OpenAI
import neo4j

In [6]:
import redis
from redis.commands.search.field import (
    NumericField,
    TagField,
    TextField,
    VectorField,
)

In [7]:
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.query import Query

In [8]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [9]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [10]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [11]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:22:03


### OpenAI

In [12]:
llm_client = OpenAI()

### Neo4j

In [13]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [14]:
session = driver.session(database="neo4j")

In [15]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

### Louvain Community Detection Algorithm

In [16]:
# Cleanup the GDS graph first
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

In [17]:
# Create a new in-memory graph
query = """

CALL gds.graph.project('ds_graph', 'Complaint', 'SIMILAR',
                        {relationshipProperties: 'similarity_score'})

"""

session.run(query)

In [18]:
# Get the results from the Louvain community detection algorithm stored in DataFrame
query = """

CALL gds.louvain.stream('ds_graph', {includeIntermediateCommunities: true})
YIELD nodeId, communityId, intermediateCommunityIds
RETURN gds.util.asNode(nodeId).consumer_complaint_narrative AS narrative, communityId as community,
intermediateCommunityIds as intermediate_community
ORDER BY community, narrative ASC

"""

community_df = my_neo4j_run_query_pandas(query)

In [19]:
# Write the results from the Louvain community detection algorithm back to the nodes in the original graph
query = """
// Run Louvain on the named GDS graph and stream results
CALL gds.louvain.stream('ds_graph', {includeIntermediateCommunities: true})
YIELD nodeId, communityId, intermediateCommunityIds

// Write the results back to the original nodes
WITH gds.util.asNode(nodeId) AS n, communityId, intermediateCommunityIds
SET n.community_id           = toInteger(communityId),
    n.intermediate_community = intermediateCommunityIds,
    n.first_intermediate     = toInteger( head(intermediateCommunityIds) )
RETURN count(*) AS nodes_updated
"""

result = session.run(query)


In [20]:
# Show the summary metrics
record = result.single()
metrics = dict(record)
print(metrics)

{'nodes_updated': 250}


In [21]:
# Based on intermediate_community
community_ids = (
    community_df['intermediate_community']
    .dropna()
    .apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
    .dropna()
    .unique()
)

In [22]:
community_df[['community']].value_counts().head(10)

community
144          121
16            45
113           36
154           22
4             19
215            7
Name: count, dtype: int64

In [23]:
community_df.head()

,narrative,community,intermediate_community
0,Buy adding a hard inquiries to my consumer rep...,4,"[4, 4]"
1,Consumer Reporting Agencies play a crucial rol...,4,"[4, 4]"
2,I am filing this complaint because a consumer ...,4,"[4, 4]"
3,I am writing to formally file a complaint rega...,4,"[4, 4]"
4,I did not approve these transactions to be inc...,4,"[4, 4]"


In [24]:
community_df['intermediate_community'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
).value_counts().head(10)

intermediate_community
144    66
151    55
16     42
154    22
20     21
4      19
113    15
215     7
242     2
77      1
Name: count, dtype: int64

### Summarize the complaints

In [25]:
# This is based on intermediate_community (first intermediate)
def make_bullet_list(df, first_id, sample_size=SUMMARY_SAMPLE, random_state=RAND_SEED):
    df = df.copy()
    df["first_intermediate"] = df["intermediate_community"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
    )
    
    # Select rows for the given community
    texts = df.loc[df["first_intermediate"] == first_id, "narrative"].dropna().astype(str)

    # Take a small random sample
    texts = texts.sample(n=min(len(texts), sample_size), random_state=random_state)

    # Convert to a bullet list
    bullet_list = "\n".join(f"- {t}" for t in texts)
    return bullet_list    

In [26]:
prompt = """
You are a helpful assistant.
You will be provided with multiple bullet points.
Each bullet point represents a consumer complaint narrative belonging to a similar complaint category.

Please read all bullet points together and write ONE short summary (ideally several words or fewer)
that best describes the main theme shared by all of them.

Note: Many summaries tend to mention "inaccurate." 
Avoid using that word if there is any more specific or meaningful theme present.

Important: Make the summary distinct from summaries of other categories.
Do NOT produce something that could easily apply to multiple categories.
Be specific, concise, and descriptive.
DO NOT include any company names, person names, or other identifying information in the summary.
"""

In [27]:
def summarize_complaint(to_be_summarized, prompt, history="", model="gpt-4o-mini", temperature=0):
    completion = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": prompt},
            {"role": "user", "content": to_be_summarized},
            {"role": "user", "content": history}
        ],
        temperature=temperature
    )

    return(completion.choices[0].message.content)

In [28]:
summarized_complaints = []
historical_answers = "Summary for other categories: "

for idx, community_id in enumerate(community_ids):
    bullet_points = make_bullet_list(community_df, community_id)

    if idx == 0:
        summarized = summarize_complaint(bullet_points, prompt)
    else:
        summarized = summarize_complaint(bullet_points, prompt, historical_answers)
    
    summary = {
        "community_id": int(community_id),
        "summary": summarized
    }

    summarized_complaints.append(summary)
    historical_answers = historical_answers + ", " + summarized


In [29]:
summarized_complaints[:10]

[{'community_id': 4,
  'summary': 'Violation of consumer privacy and inaccurate credit reporting.'},
 {'community_id': 16,
  'summary': 'Identity theft and fraudulent credit activity.'},
 {'community_id': 242,
  'summary': 'Identity theft and unauthorized use of personal information.'},
 {'community_id': 77, 'summary': 'Identity theft and its consequences.'},
 {'community_id': 113,
  'summary': 'Fraudulent accounts resulting from data breaches and identity theft.'},
 {'community_id': 20,
  'summary': 'Fraudulent accounts due to data breaches and unauthorized access.'},
 {'community_id': 144,
  'summary': 'Inaccurate credit reporting and failure to validate debts.'},
 {'community_id': 151,
  'summary': 'Failure to correct inaccuracies in credit reporting and disputes.'},
 {'community_id': 154,
  'summary': 'Unauthorized accounts and inaccuracies on credit reports.'},
 {'community_id': 215,
  'summary': 'Unauthorized accounts and failure to investigate credit report disputes.'}]

### Add Summary to Graph DB

In [ ]:
# Create constraint
query = """
CREATE CONSTRAINT category_summary_unique IF NOT EXISTS
FOR (cat:Category)
REQUIRE cat.summary IS UNIQUE
"""

session.run(query)

In [31]:
# Create Category node
# This is based on first_intermediate

# Add intermediate_community property to the nodes
query = """
MATCH (c:Complaint)
WHERE c.intermediate_community IS NOT NULL AND size(c.intermediate_community) > 0
SET c.first_intermediate = toInteger(c.intermediate_community[0]);
"""
session.run(query)

# Add Category to the Graph DB
query = """
UNWIND $rows AS row
MERGE (cat:Category {community_id: toInteger(row.community_id)})
SET cat.summary = row.summary

// Link all complaints where first_intermediate matches the category id
WITH cat
MATCH (c:Complaint)
WHERE toInteger(c.first_intermediate) = cat.community_id
MERGE (c)-[:IN_CATEGORY]->(cat);
"""

session.run(query, rows=summarized_complaints)

In [32]:
 # Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:22:49
Program elapsed time: 0 minutes and 46.40 seconds
